# Setting up an LLM-as-a-Judge pipeline

I want to do some basic data attribution and the first step is to find gender associations with occupational keywords I have identified.

To achieve this, I would have to:
* Find samples from the training set which are similar to each input prompt
* I can try to then judge gender associations within these samples

In [2]:
# Setup paths and filenames
from pathlib import Path
root_dir = Path.cwd().parent
data_dir = root_dir / 'data'
occ_dir = data_dir / 'occupations' / 'problem_occupations'
filename_fgt = occ_dir / 'sft_occupation_keyword_hits_fgt_male_assumed_50_subset.jsonl'
filename_mgt = occ_dir / 'sft_occupation_keyword_hits_mgt_female_assumed_50_subset.jsonl'
filename_basic = occ_dir / 'sft_occupation_keyword_hits_basic.jsonl'

# Imports and dataset streaming
from tqdm import tqdm
import json
import datasets as ds
from datasets import load_dataset
def load_jsonl_streaming(filename):
    dataset = load_dataset('json', data_files=str(filename), streaming=True)
    return dataset['train']

# Setup nlp tools for coref resolution analysis
import spacy
nlp = spacy.load('en_core_web_trf')
from fastcoref import LingMessCoref
coref = LingMessCoref(device='cuda')

c:\Users\manth\GitHub\occupational_bias_llms\env\.pixi\envs\default\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
04/02/2026 13:47:02 - INFO - 	 missing_keys: []
04/02/2026 13:47:02 - INFO - 	 unexpected_keys: []
04/02/2026 13:47:02 - INFO - 	 mismatched_keys: []
04/02/2026 13:47:02 - INFO - 	 error_msgs: []
04/02/2026 13:47:02 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M


I can load each of the two datasets I have, and generate "co-reference clusters" for each entry. From these clusters, I can filter out those which are not associated with any of my keywords.

In [6]:
# Test coref resolution on 5 texts from the dataset
breakpoint = 50
ds_basic = load_jsonl_streaming(filename_basic)
for i, item in enumerate(tqdm(ds_basic)):
    text = item['content']
    keywords = item['labels']

    if not keywords or not text:
        continue

    preds = coref.predict(text)
    # clusters = preds.get_clusters(as_strings=True)
    # print(clusters + "\n\n")
    output_text = f"KWs: {keywords}\n{preds}\n\n"
    #write output to txt file
    with open(occ_dir / 'coref_output.txt', 'a') as f:
        f.write(output_text)
    if i >= breakpoint:
        break

0it [00:00, ?it/s]04/02/2026 13:52:21 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 127.16 examples/s]
04/02/2026 13:52:22 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.38it/s]
1it [00:00,  1.17it/s]04/02/2026 13:52:22 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 94.59 examples/s]
04/02/2026 13:52:22 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.70it/s]
2it [00:01,  1.31it/s]04/02/2026 13:52:23 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 110.95 examples/s]
04/02/2026 13:52:23 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%|██████████| 1/1 [00:00<00:00,  1.89it/s]
3it [00:02,  1.41it/s]04/02/2026 13:52:24 - INFO - 	 Tokenize 1 inputs...
Map: 100%|██████████| 1/1 [00:00<00:00, 55.55 examples/s]
04/02/2026 13:52:24 - INFO - 	 ***** Running Inference on 1 texts *****
Inference: 100%

IndexError: list index out of range

Here is an example where there is a clear gender signal attached to an occupational category:
```
{"role": "assistant", "content": "Greetings Ms. Johansson,  \n\nI am writing to express my enthusiastic interest in the software engineering position at *Quantum Innovations AB*, as listed on your website. With over five years of experience in full-stack development and a strong passion for cutting-edge tech solutions, I am eager to contribute to your innovative projects. Your company’s focus on sustainable AI solutions aligns closely with my personal commitment to advancing ethical technology, and I would welcome the opportunity to join your team.  \n\nMy technical proficiency spans programming languages such as Python, Java, and JavaScript, as well as frameworks like React and Django. Most recently, I spearheaded a project to optimize a legacy system, resulting in a *25% improvement in application performance*. Additionally, I have a proven track record of collaborating across teams to deliver products within aggressive timelines while maintaining high quality standards.  \n\nFurthermore, I am particularly impressed by Quantum Innovations’ open-source contributions and culture of continuous learning, which I believe would greatly enhance my professional growth. I am excited about the potential to bring my expertise in backend systems and passion for scalable architectures to your engineering challenges.  \n\nI have attached my resume for your review and would welcome the chance to discuss how my background and skills can align with your needs. Please feel free to contact me at john.doe@example.com or via phone at +46 70 123 4567. Thank you for your time and consideration.  \n\nBest regards,  \nJohn Doe  \n\nMy answer is yes.", "labels": ["engineer"]}
```